# 9.3 - Fine-Tuning with Vertex AI

**Module 9 - Fine-Tuning** | Netsetos GenAI Engineering

This notebook is a hands-on tour of Google's managed tuning path for Gemini, built entirely on the current `google-genai` SDK (the old `vertexai.tuning` was removed in June 2026). Each cell is a tiny, runnable decision helper - format a dataset, launch a job, pick hyperparameters, estimate cost, call and evaluate the tuned endpoint, and choose between managed Gemini and open-weight Gemma - so you see the whole pipeline as small pieces of plain Python rather than a black box.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the one SDK this lesson depends on. Everything here targets the unified `google-genai` client - the successor to the removed `vertexai` Generative AI SDK - so the code you write has no expiry date.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai python-dotenv -q  # noqa


**Explanation**

A dependency-install cell, commented out so it does nothing unless you uncomment it on a fresh environment such as Colab. It is environment prep, not code that runs any logic.

**How the code works - step by step**
- **`google-genai`** - the current unified SDK; one `Client` object drives both the Gemini API and Vertex AI, including tuning jobs.
- **`python-dotenv`** - lets you load API keys from a `.env` file instead of hardcoding them.
- The `-q` flag quiets pip; `# noqa` marks the line so linters ignore the shell-magic `!`.

**In one line:** install the current SDK so no example here calls the removed `vertexai.tuning`.

**What you'll see:** (no output - environment prep)

## Setup - API keys from the environment

Before any client call, load your credentials from the environment rather than pasting them into code. Any one provider key is enough to start experimenting.

> **Needs a Google API key** (set `GOOGLE_API_KEY`, from aistudio.google.com) for the live Gemini/Vertex calls later in the notebook.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A configuration cell that reserves an environment variable slot for your key without overwriting one you may already have set. `setdefault` only fills the value if it is currently absent, so a real key exported in your shell wins.

**How the code works - step by step**
- **`import os`** - gives access to the process environment.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - creates the key with an empty default only if it is not already present; the inline comment points to where you get the key (aistudio.google.com).
- Nothing is hardcoded - the actual secret lives in your shell or `.env`, never in the notebook.

**In one line:** wire up the key slot safely, without clobbering an existing key.

**What you'll see:** (no output - environment prep)

## 1 - Format data the way Gemini demands

The managed pipeline is six stages (prepare JSONL -> upload to GCS -> launch -> train -> evaluate -> auto-deploy) and you only touch the ends. The end that fails jobs is the format: Gemini does NOT use OpenAI's shape. It wants `contents` with `role: "model"` (not "assistant") and `parts: [{"text": ...}]` (not a bare string).

In [ ]:
# GEMINI TUNING DATA FORMAT - role "model" (not "assistant"), parts:[{text:...}].
import json

def to_gemini(system, pairs):
    out = []
    for user, model in pairs:
        out.append({"systemInstruction": {"role": "system", "parts": [{"text": system}]},
                    "contents": [{"role": "user",  "parts": [{"text": user}]},
                                 {"role": "model", "parts": [{"text": model}]}]})
    return out

pairs = [("What is the refund policy?", "Full refund within 7 days, 50% from day 8-30, none after 30."),
         ("How much is the GenAI course?", "14,999 rupees with lifetime access.")]
data = to_gemini("You are a Netsetos support assistant.", pairs)
jsonl = "\n".join(json.dumps(x, ensure_ascii=False) for x in data)   # one example per line
print(f"  {len(data)} examples -> Gemini tuning JSONL. First line (truncated):")
print("   ", jsonl.splitlines()[0][:88], "...")
print("  Gemini vs OpenAI format:")
print("    OpenAI: 'messages', role 'assistant', content is a string")
print("    Gemini: 'contents', role 'model',     parts is [{'text': ...}]  (+ optional systemInstruction)")

# Output:
#   2 examples -> Gemini tuning JSONL. First line (truncated):
#     {"systemInstruction": {"role": "system", "parts": [{"text": "You are a Netsetos support  ...
#   Gemini vs OpenAI format:
#     OpenAI: 'messages', role 'assistant', content is a string
#     Gemini: 'contents', role 'model',     parts is [{'text': ...}]  (+ optional systemInstruction)

**Explanation**

A pure data-transform cell - no model call. It converts (user, model) pairs into Gemini's tuning JSONL schema and serializes them one example per line, then prints the exact contrast between the OpenAI and Gemini formats so the load-bearing difference is impossible to miss.

**How the code works - step by step**
- **`to_gemini(system, pairs)`** - wraps each (user, model) tuple in the Gemini schema: a `systemInstruction` block plus a `contents` list with a `role: user` turn and a `role: model` turn, each using `parts: [{"text": ...}]`.
- **`pairs`** - two toy support Q&A examples (refund policy, course price).
- **`json.dumps(..., ensure_ascii=False)`** joined by newlines - produces JSONL, one example per line, exactly what a tuning job reads from GCS.
- The final prints spell out the gotcha: OpenAI uses `messages` / `assistant` / string content; Gemini uses `contents` / `model` / `parts`.

**In one line:** turn pairs into Gemini-shaped JSONL, and show why the OpenAI format would be rejected.

**What you'll see:** Prints that 2 examples were converted, the first JSONL line truncated to 88 chars (starting with the `systemInstruction`), and a two-line side-by-side of the OpenAI vs Gemini format differences.

## 2 - Launch a tuning job on the current SDK

This is the core of the lesson: one client, one call. `client.tunings.tune(...)` hands Google your base model and GCS dataset; it provisions GPUs, runs LoRA, and auto-deploys the result. The old `vertexai.tuning.sft.train` was removed on 2026-06-24, so every example here is on the SDK that still imports.

> **Needs a Google Cloud project + Vertex AI** - a real job runs on Vertex (~1-4 hours); the cell prints an illustrative pending state.

In [ ]:
# LAUNCH A GEMINI TUNING JOB - the CURRENT google-genai SDK.
# pip install google-genai   (the old `vertexai.tuning` was REMOVED 2026-06-24)
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project="your-project", location="us-central1")

job = client.tunings.tune(
    base_model="gemini-2.5-flash",          # 2.0 is EOL; verify the current tunable lineup
    training_dataset=types.TuningDataset(gcs_uri="gs://YOUR_BUCKET/finetune/train.jsonl"),
    config=types.CreateTuningJobConfig(
        epoch_count=3,                        # was `epochs`
        learning_rate_multiplier=1.0,         # 1.0 default; Pro + small data -> 10
        adapter_size=types.AdapterSize.ADAPTER_SIZE_FOUR,   # LoRA rank; 4 is the sweet spot
        tuned_model_display_name="netsetos-support-v1",
    ),
)
print(f"Job: {job.name}  state: {job.state}")   # JOB_STATE_PENDING

# poll until done, then read the auto-deployed endpoint:
# import time
# while job.state not in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED"):
#     time.sleep(60); job = client.tunings.get(name=job.name); print(job.state)
# print("tuned endpoint:", job.tuned_model.endpoint)

# Output: (illustrative - a real job runs on Vertex AI, ~1-4 hours)
# Job: projects/.../locations/us-central1/tuningJobs/1234567890  state: JOB_STATE_PENDING

**Explanation**

The canonical launch call, shown with the polling loop commented out. It is the whole managed-tuning API in one statement, and the parameter names double as the migration cheat-sheet from the removed SDK to the current one.

**How the code works - step by step**
- **`genai.Client(vertexai=True, project=..., location=...)`** - the unified client, pointed at Vertex.
- **`client.tunings.tune(...)`** - the single launch call.
- **`base_model="gemini-2.5-flash"`** - the tunable base (renamed from the old `source_model`); the comment warns 2.0 is EOL, so verify the current lineup.
- **`training_dataset=types.TuningDataset(gcs_uri=...)`** - the JSONL location (was `train_dataset`).
- **`config=types.CreateTuningJobConfig(...)`** - holds `epoch_count` (was `epochs`), `learning_rate_multiplier`, `adapter_size` (LoRA rank as an enum, `ADAPTER_SIZE_FOUR`), and a display name.
- The commented block shows the lifecycle: poll `client.tunings.get` until `SUCCEEDED`/`FAILED`, then read `job.tuned_model.endpoint`.

**In one line:** one `tune` call launches everything; the renamed args ARE the SDK migration.

**What you'll see:** Prints one line with the job name and its state, e.g. `Job: projects/.../tuningJobs/1234567890  state: JOB_STATE_PENDING` (illustrative - a real job runs for hours on Vertex).

## 3 - Pick hyperparameters that don't overfit

Vertex exposes only a few knobs - `learning_rate_multiplier`, `epoch_count`, `adapter_size` - and auto-tunes the rest. The single most-missed rule: Flash models are fine on defaults, but a Pro model on a SMALL dataset needs an aggressive learning rate and many epochs (Google's guidance: LR x10, 20 epochs).

In [ ]:
# HYPERPARAMETERS - Flash uses defaults; Pro + a small dataset needs aggressive settings.
def recommend_hparams(model, n_examples):
    pro = "pro" in model
    small = n_examples < 1000
    if pro and small:      # Google's specific best-practice for Pro + small text data
        return dict(learning_rate_multiplier=10, epoch_count=20, adapter_size=4)
    if pro:
        return dict(learning_rate_multiplier=1.0, epoch_count=10, adapter_size=4)
    return dict(learning_rate_multiplier=1.0, epoch_count="default (auto)", adapter_size=4)   # Flash

for model, n in [("gemini-2.5-flash", 500), ("gemini-2.5-pro", 500), ("gemini-2.5-pro", 20000)]:
    print(f"  {model:18s} n={n:<6} -> {recommend_hparams(model, n)}")
print("  Rule: Flash -> defaults work; Pro + small dataset -> aggressive (LR=10, epochs=20).")

# Output:
#   gemini-2.5-flash   n=500    -> {'learning_rate_multiplier': 1.0, 'epoch_count': 'default (auto)', 'adapter_size': 4}
#   gemini-2.5-pro     n=500    -> {'learning_rate_multiplier': 10, 'epoch_count': 20, 'adapter_size': 4}
#   gemini-2.5-pro     n=20000  -> {'learning_rate_multiplier': 1.0, 'epoch_count': 10, 'adapter_size': 4}
#   Rule: Flash -> defaults work; Pro + small dataset -> aggressive (LR=10, epochs=20).

**Explanation**

A rules-of-thumb function - no model call - that encodes Google's published best-practice as branching logic keyed on the model family and dataset size. It makes the hidden rule (defaults badly underfit Pro-on-small-data) explicit and testable.

**How the code works - step by step**
- **`recommend_hparams(model, n_examples)`** - decides settings from two facts: whether the model is a Pro (`"pro" in model`) and whether the set is small (`< 1000`).
- **Pro + small** -> the aggressive recipe: `learning_rate_multiplier=10`, `epoch_count=20`, `adapter_size=4`.
- **Pro + large** -> moderate: LR 1.0, 10 epochs.
- **Flash (any size)** -> defaults: LR 1.0, epochs auto.
- **`adapter_size` stays 4** in every branch - the sweet spot; raise it only for genuinely complex patterns.
- The loop runs three cases (Flash-500, Pro-500, Pro-20000) to show the branches firing.

**In one line:** Flash -> defaults; Pro + small data -> aggressive (LR 10, 20 epochs); adapter_size 4 throughout.

**What you'll see:** Prints three recommendation rows - Flash n=500 gets defaults, Pro n=500 gets LR 10 / 20 epochs, Pro n=20000 gets LR 1.0 / 10 epochs - followed by the one-line rule.

## 4 - Cost: training scales with epochs, inference has no premium

Vertex tuning is cheap because it is LoRA, not full fine-tuning: training cost = dataset tokens x epochs x a per-million rate. The part that surprises people: there is NO inference premium - a tuned Gemini costs exactly the same per token as the base, because the adapter rides the shared base endpoint.

In [ ]:
# COST - training is tokens x epochs x rate; inference has NO tuning premium.
# illustrative Jun-2026 rates - verify current pricing.
TRAIN_PER_MTOK = 3.0            # ~ Flash training $/1M training tokens
INFER = {"flash": (0.30, 2.50), "flash-lite": (0.10, 0.40)}   # $/1M in, out (tuned = SAME as base)

def training_cost(dataset_tokens, epochs):
    return dataset_tokens * epochs / 1e6 * TRAIN_PER_MTOK

for toks, ep in [(100000, 4), (1000000, 5), (10000000, 3)]:
    print(f"  {toks:>10,} tokens x {ep} epochs -> ~${training_cost(toks, ep):,.2f} to train")

tin, tout, qpm = 500, 200, 50000
base = (tin*INFER["flash"][0] + tout*INFER["flash"][1]) / 1e6 * qpm
print(f"  Inference (tuned Flash): ${base:,.2f}/mo at {qpm:,} queries -> SAME as base model ($0 tuning premium)")
print("  Contrast: OpenAI charges a fine-tuned-inference markup; Vertex does not (adapter shares the base endpoint).")

# Output:
#      100,000 tokens x 4 epochs -> ~$1.20 to train
#    1,000,000 tokens x 5 epochs -> ~$15.00 to train
#   10,000,000 tokens x 3 epochs -> ~$90.00 to train
#   Inference (tuned Flash): $32.50/mo at 50,000 queries -> SAME as base model ($0 tuning premium)
#   Contrast: OpenAI charges a fine-tuned-inference markup; Vertex does not (adapter shares the base endpoint).

**Explanation**

A back-of-the-envelope cost calculator, not an API call. It models the two costs separately - one-time training (which scales with epochs) and ongoing inference (which does not change when tuned) - to make the no-premium point concrete against providers that do charge a markup.

**How the code works - step by step**
- **`TRAIN_PER_MTOK` / `INFER`** - illustrative Jun-2026 rates; a training $/1M-token rate and per-tier in/out inference rates (tuned equals base).
- **`training_cost(dataset_tokens, epochs)`** - the entire tuning-cost model: `tokens x epochs / 1e6 x rate`.
- The loop prints three training scenarios (100K, 1M, 10M tokens at varying epochs) to show cost scaling.
- **The inference block** - computes a monthly Flash bill at 50,000 queries and states it equals the base-model bill: $0 tuning premium.
- The closing print contrasts this with OpenAI's fine-tuned-inference markup.

**In one line:** pay a small epoch-scaled training cost once; serve forever at base-model price.

**What you'll see:** Prints three training estimates (~$1.20, ~$15.00, ~$90.00), then a tuned-Flash inference line of $32.50/mo flagged as the SAME as base (no premium), then the OpenAI-markup contrast.

## 5 - Call the tuned endpoint and prove it's better

When the job succeeds, Vertex auto-deploys the adapter and hands you an endpoint you call like any other Gemini model. Then comes the step teams skip and shouldn't: evaluate base vs tuned on a held-out set - by hand here, or with Vertex's Gen AI Eval Service.

> **Needs a deployed tuned endpoint** and a Google Cloud project - the outputs shown are illustrative.

In [ ]:
# USE + EVALUATE THE TUNED MODEL - same client, the tuned endpoint.
from google import genai
client = genai.Client(vertexai=True, project="your-project", location="us-central1")

TUNED = "projects/.../locations/us-central1/endpoints/YOUR_ENDPOINT"   # job.tuned_model.endpoint
tests = ["What is the refund policy?", "Do you offer EMI options?"]

for q in tests:
    base  = client.models.generate_content(model="gemini-2.5-flash", contents=q).text
    tuned = client.models.generate_content(model=TUNED, contents=q).text
    print(f"Q: {q}\n  base : {base[:70]}\n  tuned: {tuned[:70]}")

# Managed base-vs-tuned scoring with the Gen AI Eval Service (GA):
# result = client.evals.evaluate(dataset=eval_ds,
#             metrics=["GENERAL_QUALITY", "INSTRUCTION_FOLLOWING"])   # + BLEU / ROUGE
# print(result.summary_metrics)

# Output: (illustrative - requires a deployed tuned endpoint)
# Q: What is the refund policy?
#   base : Refund policies vary by provider; please check the website ...
#   tuned: Full refund within 7 days, 50% from day 8-30, none after 30 ...

**Explanation**

A base-vs-tuned comparison harness. The key idea is that the tuned model is just another `model=` argument to the same `generate_content` call, so evaluation is a plain loop that runs both on identical queries. The commented block points to the managed, repeatable version.

**How the code works - step by step**
- **`genai.Client(vertexai=True, ...)`** - same unified client as the launch cell.
- **`TUNED`** - the tuned endpoint path (`job.tuned_model.endpoint`), passed as `model=` just like a base model name.
- **The loop** - sends each test question to both `gemini-2.5-flash` (base) and `TUNED`, printing the first 70 chars of each answer so the difference is visible.
- **The commented `client.evals.evaluate(...)`** - the managed path: rubric metrics (`GENERAL_QUALITY`, `INSTRUCTION_FOLLOWING`) plus overlap metrics (BLEU/ROUGE) for a real scorecard.

**In one line:** call the tuned endpoint like any model, then compare it against the base on held-out questions.

**What you'll see:** For each test question, prints the base answer and the tuned answer side by side - illustratively, the base gives a vague generic reply while the tuned model returns the specific company policy (requires a real deployed endpoint to run for real).

## 6 - Managed Gemini vs open-weight Gemma

Vertex offers several tuning paths - SFT (behavior/format), DPO (preferences from chosen/rejected pairs), continuous (keep improving a tuned model), checkpoints (pick the best intermediate) - but one decision overrides them all: Gemini (managed) vs Gemma (open weights). It turns on a single question: do you need to download the trained weights?

In [ ]:
# GEMINI (managed) vs GEMMA (open weights) - the lock-in decision.
def pick_platform(need_downloadable_weights, need_on_prem_or_multicloud, want_zero_gpu_ops):
    if need_downloadable_weights or need_on_prem_or_multicloud:
        return ("Gemma (Model Garden)",
                "open weights: full LoRA/QLoRA, download and deploy anywhere - but you manage the GPUs/serving.")
    if want_zero_gpu_ops:
        return ("Gemini SFT/DPO (managed)",
                "upload JSONL, click train, auto-deployed endpoint - but weights stay on Google's shared endpoint.")
    return ("Gemini SFT/DPO (managed)", "default: the zero-ops path unless you specifically need the weights.")

for label, args in [("Enterprise, on-prem required", (True, True, False)),
                    ("Startup, no GPU team", (False, False, True)),
                    ("Multi-cloud, data sovereignty", (True, True, False))]:
    plat, why = pick_platform(*args)
    print(f"  {label:32s} -> {plat}")
    print(f"      {why}")

# Output:
#   Enterprise, on-prem required     -> Gemma (Model Garden)
#       open weights: full LoRA/QLoRA, download and deploy anywhere - but you manage the GPUs/serving.
#   Startup, no GPU team             -> Gemini SFT/DPO (managed)
#       upload JSONL, click train, auto-deployed endpoint - but weights stay on Google's shared endpoint.
#   Multi-cloud, data sovereignty    -> Gemma (Model Garden)
#       open weights: full LoRA/QLoRA, download and deploy anywhere - but you manage the GPUs/serving.

**Explanation**

A decision function, not a model call. It reduces the platform choice to the three conditions that force open weights, returning both a recommendation and a plain-English reason that names the tradeoff each way (convenience vs control).

**How the code works - step by step**
- **`pick_platform(need_downloadable_weights, need_on_prem_or_multicloud, want_zero_gpu_ops)`** - the router.
- **Downloadable weights OR on-prem/multi-cloud** -> **Gemma (Model Garden)**: full LoRA/QLoRA, deploy anywhere, but you run the GPUs and serving.
- **Zero-GPU-ops (and no weight need)** -> **Gemini managed**: upload JSONL, click train, auto-deployed endpoint, but weights stay on Google's endpoint.
- **Default** -> managed Gemini unless you specifically need the weights.
- The loop runs three profiles (on-prem enterprise, no-GPU startup, multi-cloud sovereignty) to show the branch each takes.

**In one line:** only a real need for the weights forces Gemma; everything else favors managed Gemini.

**What you'll see:** Prints three labeled cases: on-prem and multi-cloud/sovereignty route to Gemma (Model Garden), the no-GPU startup routes to managed Gemini SFT/DPO - each with a one-line reason underneath.

## 7 - Deployment auto-happens; the constraints decide fit

On success Vertex deploys for you - the adapter is registered and served on a SHARED public endpoint, no manual step. Convenient, but the constraints are the real content: no dedicated/private endpoints for tuned Gemini, no SLA (tuning is not a Covered Service), and JSON-mode can degrade a tuned model.

In [ ]:
# DEPLOYMENT CONSTRAINTS - what managed tuning can and cannot do.
def deploy_check(need_dedicated_endpoint, need_sla, using_json_mode):
    flags = []
    if need_dedicated_endpoint:
        flags.append("BLOCK: tuned Gemini serves ONLY on a shared public endpoint (no dedicated/private).")
    if need_sla:
        flags.append("BLOCK: tuning is not a Covered Service - no uptime SLA.")
    if using_json_mode:
        flags.append("WARN: controlled generation (JSON mode) can degrade a tuned model - test it.")
    return flags or ["OK: within Vertex managed-tuning constraints - auto-deployed, ready to call."]

for label, args in [("Public chatbot", (False, False, False)),
                    ("Bank needing SLA + private endpoint", (True, True, False)),
                    ("Structured-output extractor", (False, False, True))]:
    print(f"  {label}:")
    for f in deploy_check(*args):
        print("    ", f)

# Output:
#   Public chatbot:
#      OK: within Vertex managed-tuning constraints - auto-deployed, ready to call.
#   Bank needing SLA + private endpoint:
#      BLOCK: tuned Gemini serves ONLY on a shared public endpoint (no dedicated/private).
#      BLOCK: tuning is not a Covered Service - no uptime SLA.
#   Structured-output extractor:
#      WARN: controlled generation (JSON mode) can degrade a tuned model - test it.

**Explanation**

A constraint-checker function that maps three common enterprise requirements onto Vertex's managed-tuning limits. It distinguishes hard blocks from soft warnings, so you learn what the platform simply cannot do before promising it to a client.

**How the code works - step by step**
- **`deploy_check(need_dedicated_endpoint, need_sla, using_json_mode)`** - accumulates flags.
- **`need_dedicated_endpoint`** -> BLOCK: tuned Gemini serves only on a shared public endpoint.
- **`need_sla`** -> BLOCK: tuning is not a Covered Service, so no uptime SLA.
- **`using_json_mode`** -> WARN (not a block): controlled generation can degrade a tuned model - test it.
- **No flags** -> an OK message: within managed-tuning constraints, auto-deployed and ready to call.
- The loop runs three cases (public chatbot, bank needing SLA + private endpoint, structured-output extractor).

**In one line:** shared endpoint only + no SLA are hard blocks; JSON mode is a warning to test.

**What you'll see:** Prints three cases: the public chatbot returns OK, the bank returns two BLOCK lines (shared endpoint, no SLA), and the structured-output extractor returns a WARN about JSON mode degrading quality.

## 8 - India: the region gap and DPDP

Inference runs from asia-south1 (Mumbai) for all Gemini tiers, but tuning is confirmed only in us-central1 / europe-west4. So the practical pattern is tune in us-central1, serve from asia-south1 - which means training data crosses borders, and under the DPDP Act that needs deliberate safeguards.

In [ ]:
# INDIA - the region gap: inference in Mumbai, tuning crosses borders. Plan for DPDP.
def india_plan(has_pii, rbi_regulated):
    plan = ["tune in us-central1 (or europe-west4); serve inference from asia-south1 (Mumbai) for low latency"]
    if has_pii:
        plan.append("remove PII with Presidio BEFORE upload (lesson 9.2)")
        plan += ["enable CMEK", "enable VPC Service Controls", "enable Zero Data Retention", "run a DPDP DPIA"]
    if rbi_regulated:
        plan.append("for full data sovereignty, prefer Gemma on IndiaAI GPUs - training never leaves India")
    return plan

print("  Indian bank support bot (PII, RBI-regulated):")
for i, step in enumerate(india_plan(has_pii=True, rbi_regulated=True), 1):
    print(f"    {i}. {step}")
print("  Region gap: inference is in Mumbai, but tuning crosses borders to us-central1 - plan for DPDP.")

# Output:
#   Indian bank support bot (PII, RBI-regulated):
#     1. tune in us-central1 (or europe-west4); serve inference from asia-south1 (Mumbai) for low latency
#     2. remove PII with Presidio BEFORE upload (lesson 9.2)
#     3. enable CMEK
#     4. enable VPC Service Controls
#     5. enable Zero Data Retention
#     6. run a DPDP DPIA
#     7. for full data sovereignty, prefer Gemma on IndiaAI GPUs - training never leaves India
#   Region gap: inference is in Mumbai, but tuning crosses borders to us-central1 - plan for DPDP.

**Explanation**

A compliance-planner function that assembles an ordered action list from two facts about the workload. It always starts with the region pattern, then layers on DPDP safeguards for PII and a sovereignty escape hatch for regulated data - turning the region gap into a concrete checklist.

**How the code works - step by step**
- **`india_plan(has_pii, rbi_regulated)`** - builds a step list.
- **Base step (always)** - tune in us-central1 (or europe-west4), serve inference from asia-south1 for low latency.
- **`has_pii`** - prepend Presidio PII removal before upload (Lesson 9.2), then add CMEK, VPC Service Controls, Zero Data Retention, and a DPDP DPIA.
- **`rbi_regulated`** - add the sovereignty option: Gemma on IndiaAI GPUs so training never leaves India.
- The call runs the strictest case (PII + RBI-regulated) and prints the numbered plan.

**In one line:** tune abroad / serve in Mumbai, wrap PII in DPDP safeguards, and fall back to Gemma-in-India for full sovereignty.

**What you'll see:** Prints a 7-step numbered plan for an Indian bank support bot - region pattern, Presidio PII removal, CMEK, VPC-SC, Zero Data Retention, DPDP DPIA, and the Gemma/IndiaAI sovereignty option - followed by the region-gap summary line.

Across eight small cells you walked the full managed-tuning loop: shape data into Gemini's `contents`/`role: model`/`parts` JSONL, launch a job with `client.tunings.tune` on the current `google-genai` SDK, choose hyperparameters that avoid overfitting, estimate the (LoRA-cheap, no-inference-premium) cost, call and evaluate the tuned endpoint, and decide managed Gemini vs open-weight Gemma - plus deployment constraints and the India/DPDP region gap. The through-line is that a managed platform trades control for convenience: you speak its exact dialect and live on its deprecation calendar, and in return you never touch a GPU. Next, Lesson 9.4 takes the open-weight path (Gemma with Axolotl/Unsloth/TRL) where you actually get the weights, and 9.5 rebuilds the LoRA/QLoRA mechanics Vertex ran for you here by hand.